# 03 - Agrupamento de jogadores (K-Means + PCA)

Neste notebook vamos agrupar jogadores com características estatísticas
parecidas, usando **K-Means**, e visualizar esses grupos em duas dimensões
usando **PCA**.

Passo a passo:
1. Carregar dados já limpos.
2. Selecionar jogadores com minutos mínimos.
3. Selecionar as features do clustering.
4. (As métricas por 90 minutos já são criadas na limpeza.)
5. Tratar valores ausentes.
6. Normalizar com StandardScaler.
7. Testar diferentes valores de K.
8. Elbow Method.
9. Silhouette Score.
10. Rodar o K-Means.
11. Aplicar PCA.
12. Visualizar.
13. Interpretar as características de cada cluster.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_players_data
from src.data_cleaning import prepare_players_data, filter_by_minutes
from src.clustering import (
    prepare_clustering_data,
    scale_features,
    find_optimal_clusters,
    perform_kmeans,
    apply_pca,
    summarize_clusters,
)
from src.config import CLUSTER_FEATURES

sns.set_style("whitegrid")

## 1. Carregar dados já limpos

In [ ]:
df = prepare_players_data(load_players_data("../data/raw/mls_players.csv"))
df.shape

## 2. Selecionar jogadores com minutos mínimos

Jogadores com poucos minutos têm estatísticas por 90 minutos pouco
confiáveis (uma amostra pequena pode gerar números "explosivos" só por
acaso). Por isso, filtramos por um mínimo de minutos antes de agrupar.

In [ ]:
df_filtrado = filter_by_minutes(df, minimum_minutes=500)
print(f"Jogadores com 500+ minutos: {len(df_filtrado)}")

## 3. Selecionar as features do clustering

O CSV usado é a tabela "Standard Stats" do FBref -- ele **não** possui
estatísticas de criação de jogo (passes progressivos, xAG), progressão
(conduções progressivas) ou defesa (desarmes, interceptações). Por isso,
as features usadas aqui (definidas em `src/config.py`) são as que
realmente existem: produção ofensiva e disciplina, ambas por 90 minutos.

In [ ]:
CLUSTER_FEATURES

## 4 e 5. Preparar os dados (seleção de features + tratamento de ausentes)

In [ ]:
df_prontos, X, features_usadas = prepare_clustering_data(df_filtrado, CLUSTER_FEATURES)

print("Features realmente utilizadas:", features_usadas)
print("Formato da matriz X:", X.shape)
X.head()

## 6. Normalizar com StandardScaler

O K-Means usa distância entre pontos para formar os grupos. Sem
normalizar, uma feature com escala maior dominaria o cálculo. O
StandardScaler coloca todas as features na mesma escala (média 0, desvio
padrão 1).

In [ ]:
X_normalizado, scaler = scale_features(X)
X_normalizado[:5]

## 7, 8 e 9. Testar diferentes valores de K (Elbow Method + Silhouette Score)

In [ ]:
resultados_k = find_optimal_clusters(X_normalizado, max_clusters=10)
resultados_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(resultados_k["k"], resultados_k["inertia"], marker="o", color="#1f6feb")
axes[0].set_title("Elbow Method (Inércia)")
axes[0].set_xlabel("Número de clusters (K)")
axes[0].set_ylabel("Inércia")

axes[1].plot(resultados_k["k"], resultados_k["silhouette_score"], marker="o", color="#1f6feb")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Número de clusters (K)")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

**Como interpretar:** no Elbow Method procuramos o ponto onde a curva "dobra"
e para de cair tão rápido (o "cotovelo"). No Silhouette Score, valores mais
altos (mais próximos de 1) indicam clusters mais bem separados. Essas
métricas **ajudam** a escolher K, mas a decisão final também depende de o
resultado fazer sentido ao interpretarmos os grupos.

## 10. Rodar o K-Means com o K escolhido

In [ ]:
NUMERO_DE_CLUSTERS = 4  # pode ser ajustado com base no gráfico acima

rotulos, modelo_kmeans = perform_kmeans(X_normalizado, n_clusters=NUMERO_DE_CLUSTERS)

df_prontos = df_prontos.copy()
df_prontos["cluster"] = rotulos
df_prontos[["player", "squad", "pos", "cluster"] + features_usadas].head()

## 11. Aplicar PCA (apenas para visualização em 2D)

In [ ]:
componentes_pca = apply_pca(X_normalizado)
df_prontos["PC1"] = componentes_pca["PC1"]
df_prontos["PC2"] = componentes_pca["PC2"]
df_prontos[["player", "PC1", "PC2", "cluster"]].head()

## 12. Visualizar os clusters no plano PC1 x PC2

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
paleta = sns.color_palette("tab10", NUMERO_DE_CLUSTERS)
sns.scatterplot(
    data=df_prontos, x="PC1", y="PC2", hue="cluster", palette=paleta, s=60, ax=ax
)
ax.set_title("Jogadores agrupados por semelhança estatística (PCA)")
plt.legend(title="Cluster")
plt.show()

## 13. Interpretar as características de cada cluster

**Importante:** o algoritmo não sabe o que é "atacante" ou "zagueiro" --
ele só enxerga números. A interpretação de cada grupo (o que ele
representa) é feita por nós, olhando as médias e as posições mais
frequentes em cada cluster.

In [ ]:
resumo_clusters = summarize_clusters(df_prontos, features_usadas)
resumo_clusters

Observando a tabela acima, para cada cluster podemos ler:

- a **média de gols/assistências por 90 minutos** indica o quão ofensivo é o grupo;
- a **média de cartões por 90 minutos** indica o quão "duro"/disciplinar é o grupo;
- as **posições mais frequentes** ajudam a confirmar (ou não) a leitura acima.

Por exemplo, um cluster com média alta de `gls_per90` e posições
predominantemente `FW` sugere um grupo de jogadores com perfil mais
finalizador. Já um cluster com médias baixas em todas as métricas
ofensivas e predominância de `DF` sugere um perfil mais defensivo --
mas essa é uma leitura feita a partir dos dados, não uma verdade
automática do algoritmo.

## Conclusão

- O K-Means conseguiu separar jogadores por padrões de produção ofensiva e
  disciplina, mesmo com um conjunto pequeno de features.
- O gráfico de PCA mostra a separação dos grupos em duas dimensões, mas
  lembre-se: PC1 e PC2 são combinações matemáticas das features
  originais, não têm um significado "óbvio" por si só.
- Este clustering está limitado pelas colunas disponíveis no CSV -- com
  um dataset mais completo do FBref (incluindo passes, progressão e
  defesa), os grupos tenderiam a refletir estilos de jogo mais ricos.